In [1]:
from typing import Any, List
from dotenv import load_dotenv
import json
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper

## Building Agent Class


In [8]:
class Agent:
    def __init__(
        self,
        role: str = "You are a helpful assistant.",
        instructions: str = "You are a helpful assistant.",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[tool] = None,
    ):
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools if tools is not None else []

        load_dotenv()

        self.llm = LLM(
            model=self.model,
            temperature=self.temperature,
            tools=self.tools,
        )

    def invoke(self, user_message: str) -> str:
        messages = [
            SystemMessage(
                content=(
                    f"You are an AI agent with the role: {self.role}. "
                    f"Your instructions are: {self.instructions}."
                )
            )
        ]

        messages.append(UserMessage(content=user_message))

        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        while ai_message.tool_calls:
            # 1. This converts the Python object back to a raw Dictionary/JSON
            print("--- RAW JSON FROM LLM ---")
            print(json.dumps(ai_message.model_dump(), indent=2))
            print("-------------------------")
            
            for call in ai_message.tool_calls:
                function_name = call.function.name
                function_args = json.loads(call.function.arguments)
                tool_call_id = call.id

                tool = next((t for t in self.tools if t.name == function_name), None)
                if tool:
                    result = tool(**function_args)
                    messages.append(
                        ToolMessage(
                            content=json.dumps(result),
                            tool_call_id=tool_call_id,
                            name=function_name,
                        )
                    )

            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)

        # for m in messages:
        #     print(m)

        return ai_message.content

## Testing Your Agent

Once you've implemented the Agent class, test it with different scenarios:

In [9]:
agent = Agent(role="Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.


In [10]:
@tool
def calculate(expression: str) -> float:
    #Evaluate a mathematical expression"""
    return eval(expression)

In [11]:
# Create an agent with the calculator tool
math_agent = Agent(
    role="Math Assistant",
    tools=[calculate]

)

In [12]:
response = math_agent.invoke("What is 23 * 45?")
print(response)

--- RAW JSON FROM LLM ---
{
  "content": null,
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_u0PmHljbAX2OOseiYoGlT3Uu",
      "function": {
        "arguments": "{\"expression\":\"23 * 45\"}",
        "name": "calculate"
      },
      "type": "function"
    }
  ]
}
-------------------------
The result of \( 23 \times 45 \) is 1035.


In [7]:
# Test multiple tool usage
response = math_agent.invoke("If I multiply 3 by 5, what do I get? Then later add 7")
print(response)

If you multiply 3 by 5, you get 15. Then, if you add 7 to that, the result is 22.


In [14]:
@tool
def get_games(num_games: int = 1, top: bool = True) -> list[dict[str, Any]]:
    """
    Returns the top or bottom N games with highest or lowest scores.

    args:
    num_games (int): Number of games to return (default is 1)
    top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88},
    ]

    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x["Score"], reverse=top)

    # Return the N games
    return sorted_games[:num_games]

In [ ]:
# Create an agent with the multiple tools
data_analyst_agent = Agent(
role="Game Stats Assistant",
instructions="You can bring insights about a game dataset based on users questions",
tools=[get_games]

)

In [ ]:
response = data_analyst_agent. invoke("What's the best game in the dataset?")
print(response)

In [ ]:
response = data_analyst_agent.invoke("What's the worst game in the dataset?")
print(response)